# K-means

Run K-means from supplied points and initial centroids, with all iteration steps visible.

Run the setup cell first. Each numbered section has editable inputs followed by a
calculation cell. Run that section again after changing its inputs, or use
**Restart Kernel and Run All Cells** to run the examples from a clean state.
Run each workflow's calculation before its result-detail cells. Every input has
a comment describing its format or choices. Optional fields accept `None`,
`"None"`, or blank text. Rerun setup after updating these files.

Matrices accept `1 2; 3 4`, rows on separate lines, fractions such as `1/3`,
nested lists, and NumPy arrays. Rows usually represent samples and columns features.
Results shown with `cache=cache` are saved automatically for use in any notebook
through `cache.load("name")`. The [Cache notebook](09_cache.ipynb) manages them.

[Setup and notebook index](README.md)

In [ ]:
from pathlib import Path
import sys

# Works when Jupyter starts in this folder, the repository root, or a subfolder.
PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "CommonUtils.py").is_file() and (p / "notebooks" / "notebook_tools.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Start Jupyter from the ee2211-solver repository or its notebooks folder.")
for folder in (PROJECT_ROOT, PROJECT_ROOT / "notebooks"):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))

import numpy as np
import pandas as pd
import sympy as sp
from IPython.display import Markdown, display
import importlib
import notebook_tools
importlib.reload(notebook_tools)
from notebook_tools import (NotebookCache, matrix, optional_matrix, is_blank,
    number, integer, boolean, variable_names, coordinates, choice, show, show_metrics)

pd.set_option("display.precision", 10)
cache = NotebookCache()
print(f"Python {sys.version.split()[0]} | shared cache: {cache.path}")

## 1. Inputs
Each row is a point or centroid, with matching feature counts. The number of
initial centroid rows determines k. Set `max_iterations = None` to continue until
convergence, or a positive integer to limit updates.

Distances are Euclidean; cost is the sum of squared assigned distances. Ties
choose the first centroid. A cluster with no members keeps its previous centroid.
Displayed cluster numbers start at 1; raw `assignments` in history start at 0.

In [ ]:
X = '1 1; 1.5 2; 5 7; 6 8'  # Point rows by feature columns; numeric matrix.
centroids = '1 1; 5 7'  # One centroid per row; same feature columns as X.
max_iter = None  # Positive update limit; None continues to convergence.
show_iterations = True  # True shows every step; False shows only the summary.
show_distance_matrices = False  # True shows point-to-centroid distances; False hides them.

In [ ]:
from KMeansUtils import k_means

points, initial_centroids = matrix(X), matrix(centroids)
kmeans_history = k_means(points, initial_centroids, integer(max_iter, optional=True))
final = kmeans_history[-1]
show_metrics({"Points": points.shape[0], "Features": points.shape[1], "Clusters": initial_centroids.shape[0],
    "Updates": final["iteration"], "Converged": final["converged"], "Final cost": final["cost"]})
show("final_centroids", final["centroids"], cache=cache, source="K-means")
show("clusters", (final["assignments"]+1).reshape(-1, 1), cache=cache, source="K-means (1-based clusters)")
display(pd.DataFrame([{key: step[key] for key in ("iteration", "cost", "converged")} for step in kmeans_history]))

## 2. Iteration details
Iteration zero assigns points to the supplied initial centroids. Later entries
show the updated centroids and assignments. If an iteration limit is reached,
the final state may still be unconverged.

In [ ]:
if boolean(show_iterations):
    for step in kmeans_history:
        display(Markdown(f"### Iteration {step['iteration']}"))
        print(f"Cost: {step['cost']:.10g}; converged: {step['converged']}")
        show("centroids", step["centroids"])
        display(pd.DataFrame({"Point (1-based)": np.arange(1, len(points)+1), "Cluster (1-based)": step["assignments"]+1}))
        if boolean(show_distance_matrices):
            show("point_to_centroid_distances", step["distances"])